## *Implementing a stream processing application*

This streaming application simulates real time processing cluster using Spark Structured Streaming. Although the dataset is historical, we replay task usage events incrementally by reading CSV files as a stream.
Task start timestamps are converted into Spark timestamps. We compute the average CPU usage over the entire cluster using sliding windows of 5 minutes updated every minute. The results are continuously produced every 5 seconds.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, window, avg
import time

# initialize the stream processing
spark = (
    SparkSession.builder
    .appName("GoogleClusterStreamingSimulation")
    .master("local[*]")     # uses all available cores
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")     # reduce log level

# the structure of the incoming data and the data type
schema = """start LONG,end LONG,job_id STRING,task_idx INT,machine_id STRING,cpu FLOAT,ram FLOAT"""

# simulating real time with stream
task_usage_stream = (
    spark.readStream        # reading data as a stream
    .schema(schema)         # according to the schema
    .option("maxFilesPerTrigger", 1)    # reads csv files one by one
    .csv("./data/task_usage/")
)

# converts the start timestamp from microseconds to Spark timestamp (event_time column)
task_usage_stream = task_usage_stream.withColumn("event_time", (col("start") / 1_000_000).cast("timestamp"))

# sliding window aggregation
cpu_windowed = (
    task_usage_stream
    .groupBy( window(col("event_time"), "5 minutes", "1 minute") )  # groups data by windows of 5 minutes, slides of 1 minute
    .agg(avg("cpu").alias("avg_cpu"))   # computes average CPU usage accross the cluster
)                                       # [t, t+5min] , avg_cpu

# outputs the results continuosly
query = (
    cpu_windowed
    .writeStream
    .outputMode("complete")     # reprints the full window state every time
    .format("console")          # output to console
    .option("truncate", False)
    .trigger(processingTime="5 seconds")    # starts a new micro batch every 5 seconds, if the previous one finished
    .start()          
)

query.awaitTermination(60)
query.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/15 01:21:55 WARN Utils: Your hostname, im2ag-mandelbrot, resolves to a loopback address: 127.0.1.1; using 152.77.81.20 instead (on interface ens18)
26/01/15 01:21:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/15 01:21:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/15 01:21:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------------------------------+--------------------+
|window                                    |avg_cpu             |
+------------------------------------------+--------------------+
|{1970-01-04 13:25:00, 1970-01-04 13:30:00}|0.019739761694363412|
|{1970-01-04 13:50:00, 1970-01-04 13:55:00}|0.020896184724762568|
|{1970-01-04 12:47:00, 1970-01-04 12:52:00}|0.02169524436632319 |
|{1970-01-04 13:13:00, 1970-01-04 13:18:00}|0.01941146307072241 |
|{1970-01-04 13:49:00, 1970-01-04 13:54:00}|0.020983218885484   |
|{1970-01-04 12:37:00, 1970-01-04 12:42:00}|0.01160982050088025 |
|{1970-01-04 13:59:00, 1970-01-04 14:04:00}|0.02147850064250408 |
|{1970-01-04 12:54:00, 1970-01-04 12:59:00}|0.018929027699395914|
|{1970-01-04 13:16:00, 1970-01-04 13:21:00}|0.018379244950938897|
|{1970-01-04 13:18:00, 1970-01-04 13:23:00}|0.018112656635893305|
|{1970-01-04 13:08:00, 1970-01-04 13:13:00}|0

-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+--------------------+
|window                                    |avg_cpu             |
+------------------------------------------+--------------------+
|{1970-01-04 14:08:00, 1970-01-04 14:13:00}|0.01803269153952746 |
|{1970-01-04 13:25:00, 1970-01-04 13:30:00}|0.019739761694363412|
|{1970-01-04 14:40:00, 1970-01-04 14:45:00}|0.022447996456636067|
|{1970-01-04 13:50:00, 1970-01-04 13:55:00}|0.020896184724762568|
|{1970-01-04 12:47:00, 1970-01-04 12:52:00}|0.02169524436632319 |
|{1970-01-04 14:13:00, 1970-01-04 14:18:00}|0.02346605177485824 |
|{1970-01-04 14:52:00, 1970-01-04 14:57:00}|0.022057669874939678|
|{1970-01-04 14:16:00, 1970-01-04 14:21:00}|0.023731004799403484|
|{1970-01-04 13:13:00, 1970-01-04 13:18:00}|0.01941146307072241 |
|{1970-01-04 13:49:00, 1970-01-04 13:54:00}|0.020983218885484   |
|{1970-01-04 12:37:00, 1970-01-04 12:42:00}|0

-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+--------------------+
|window                                    |avg_cpu             |
+------------------------------------------+--------------------+
|{1970-01-04 16:30:00, 1970-01-04 16:35:00}|0.02387213442971031 |
|{1970-01-04 13:25:00, 1970-01-04 13:30:00}|0.019739761694363412|
|{1970-01-04 14:08:00, 1970-01-04 14:13:00}|0.01803269153952746 |
|{1970-01-04 14:40:00, 1970-01-04 14:45:00}|0.022447996456636067|
|{1970-01-04 13:50:00, 1970-01-04 13:55:00}|0.020896184724762568|
|{1970-01-04 16:18:00, 1970-01-04 16:23:00}|0.02467560515307166 |
|{1970-01-04 15:52:00, 1970-01-04 15:57:00}|0.022021051243482352|
|{1970-01-04 14:13:00, 1970-01-04 14:18:00}|0.02346605177485824 |
|{1970-01-04 14:52:00, 1970-01-04 14:57:00}|0.022057669874939678|
|{1970-01-04 12:47:00, 1970-01-04 12:52:00}|0.02169524436632319 |
|{1970-01-04 14:16:00, 1970-01-04 14:21:00}|0

-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+--------------------+
|window                                    |avg_cpu             |
+------------------------------------------+--------------------+
|{1970-01-04 16:30:00, 1970-01-04 16:35:00}|0.02387213442971031 |
|{1970-01-04 14:08:00, 1970-01-04 14:13:00}|0.01803269153952746 |
|{1970-01-04 13:25:00, 1970-01-04 13:30:00}|0.019739761694363412|
|{1970-01-04 17:19:00, 1970-01-04 17:24:00}|0.02265302545923963 |
|{1970-01-04 14:40:00, 1970-01-04 14:45:00}|0.022447996456636067|
|{1970-01-04 13:50:00, 1970-01-04 13:55:00}|0.020896184724762568|
|{1970-01-04 17:54:00, 1970-01-04 17:59:00}|0.02282456544890567 |
|{1970-01-04 16:18:00, 1970-01-04 16:23:00}|0.02467560515307166 |
|{1970-01-04 17:15:00, 1970-01-04 17:20:00}|0.02255709830582908 |
|{1970-01-04 15:52:00, 1970-01-04 15:57:00}|0.022021051243482352|
|{1970-01-04 14:13:00, 1970-01-04 14:18:00}|0

-------------------------------------------
Batch: 4
-------------------------------------------
+------------------------------------------+--------------------+
|window                                    |avg_cpu             |
+------------------------------------------+--------------------+
|{1970-01-04 16:30:00, 1970-01-04 16:35:00}|0.02387213442971031 |
|{1970-01-04 14:08:00, 1970-01-04 14:13:00}|0.01803269153952746 |
|{1970-01-04 13:25:00, 1970-01-04 13:30:00}|0.019739761694363412|
|{1970-01-04 18:44:00, 1970-01-04 18:49:00}|0.02356256800730846 |
|{1970-01-04 17:19:00, 1970-01-04 17:24:00}|0.02265302545923963 |
|{1970-01-04 14:40:00, 1970-01-04 14:45:00}|0.022447996456636067|
|{1970-01-04 13:50:00, 1970-01-04 13:55:00}|0.020896184724762568|
|{1970-01-04 17:54:00, 1970-01-04 17:59:00}|0.02282456544890567 |
|{1970-01-04 16:18:00, 1970-01-04 16:23:00}|0.02467560515307166 |
|{1970-01-04 17:15:00, 1970-01-04 17:20:00}|0.02255709830582908 |
|{1970-01-04 15:52:00, 1970-01-04 15:57:00}|0